<a href="https://colab.research.google.com/github/manojgundumogula55-commits/Digital-Library-Usage-And-Resource-Analytics/blob/main/Digital_Library_Usage_Analytics/%2002_Data_Extraction_Validation_and_Cleaning/%20Data_Extraction_Validation_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Digital Library Usage Analysis
## Data Extraction, Validation and Cleaning

In [2]:
## 1. Import Libraries

In [3]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [4]:
## 2. Load the Raw Dataset

In [5]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Saving library_usage_raw.csv to library_usage_raw.csv
Dataset loaded successfully.
Rows: 1500
Columns: 15


In [6]:
# 3. Data Extraction
### The extraction tables below organize the information needed for the later analysis stage. They are views of the raw data, not separate cleaned datasets.

In [9]:
## 3.1 Resource-based Extraction

In [11]:
resource_data = df[
    ["access_id", "resource_id", "resource_title",
     "resource_type", "publisher", "publication_year"]
].copy()

display(resource_data.head(10))

,access_id,resource_id,resource_title,resource_type,publisher,publication_year
0,A00001,R070,SpringerLink Vol.1,Database,IEEE,2015
1,A00002,R068,PhD Thesis Vol.3,Thesis,Wiley,2008
2,A00003,R023,Finance Vol.2,Journal,Pearson,2005
3,A00004,R049,Finance Vol.2,Journal,NPTEL,2011
4,A00005,R056,Signal Processing Vol.5,Journal,Wiley,2021
5,A00006,R070,SpringerLink Vol.1,Database,IEEE,2015
6,A00007,R055,Data Science Vol.1,E-Book,Springer,2016
7,A00008,R038,Marketing Vol.2,E-Book,Taylor & Francis,2017
8,A00009,R036,JSTOR Vol.3,Database,McGraw Hill,2010
9,A00010,R038,Marketing Vol.2,E-Book,Taylor & Francis,2017


In [8]:
### Resource-wise Access Summary

In [10]:
resource_summary = (
    df.groupby(["resource_id", "resource_title", "resource_type"], dropna=False)
      .agg(
          access_count=("access_id", "count"),
          unique_users=("user_id", "nunique"),
          departments=("department", lambda x: ", ".join(sorted(x.dropna().astype(str).str.strip().str.upper().unique())))
      )
      .reset_index()
      .sort_values("access_count", ascending=False)
)

display(resource_summary.head(20))

,resource_id,resource_title,resource_type,access_count,unique_users,departments
62,R070,SpringerLink Vol.1,Database,129,114,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
59,R067,Data Science Vol.2,E-Book,93,85,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
32,R038,Marketing Vol.2,E-Book,84,76,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
10,R013,Marketing Vol.3,E-Book,75,69,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
65,R073,Thermodynamics Vol.1,E-Book,63,59,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
55,R063,Chemistry Vol.1,Journal,57,51,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
30,R036,JSTOR Vol.3,Database,54,50,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
15,R019,ScienceDirect Vol.4,Database,52,52,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
35,R042,Finance Vol.1,Journal,50,48,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"
19,R023,Finance Vol.2,Journal,49,45,"CIVIL, CSE, ECE, MANAGEMENT, MECHANICAL, SCIENCES"


In [12]:
### Resources Used by Multiple Departments

In [15]:
dept_by_resource = (
    df.dropna(subset=["resource_id", "department"])
      .assign(department=lambda d: d["department"].str.strip().str.upper())
      .groupby("resource_id")["department"]
      .nunique()
      .reset_index(name="department_count")
)

multi_dept_resources = dept_by_resource[dept_by_resource["department_count"] > 1]
display(multi_dept_resources)

,resource_id,department_count
0,R001,5
1,R002,6
3,R005,6
4,R006,4
5,R007,3
...,...,...
66,R075,4
67,R076,6
68,R077,5
69,R078,3


In [14]:
## 3.2 User and Department-based Extraction

In [13]:
user_data = df[
    ["access_id", "user_id", "user_type", "department", "resource_type"]
].copy()

user_summary = (
    user_data.groupby(["department", "user_type"], dropna=False)
    .size()
    .reset_index(name="access_count")
    .sort_values(["department", "user_type"])
)

display(user_summary)

,department,user_type,access_count
0,cse,Faculty,7
1,cse,Postgraduate,7
2,cse,Undergraduate,15
3,CSE,Faculty,30
4,CSE,Postgraduate,56
5,CSE,Researcher,32
6,CSE,UNDERGRADUATE,2
7,CSE,Undergraduate,145
8,Civil,FACULTY,1
9,Civil,Faculty,21


In [19]:
## Faculty and Researcher Accesses

In [18]:
faculty_researcher = user_data[
    user_data["user_type"].astype(str).str.strip().str.lower().isin(["faculty", "researcher"])
]
display(faculty_researcher)

,access_id,user_id,user_type,department,resource_type
2,A00003,U1015,Faculty,Civil,Journal
5,A00006,U1026,Faculty,Mechanical,Database
8,A00009,U1099,Faculty,cse,Database
11,A00012,U1063,Faculty,Civil,Database
14,A00015,U1442,Faculty,CSE,Journal
...,...,...,...,...,...
1481,A01482,U1678,Faculty,CSE,Database
1489,A01490,U1221,Faculty,CSE,E-Book
1492,A01493,U1134,Faculty,Mechanical,E-Book
1495,A01496,U1038,Faculty,Management,E-Book


In [16]:
## 3.3 Time-based Extraction

In [17]:
time_data = df[["access_id", "access_datetime", "resource_type", "session_minutes"]].copy()
time_data["access_datetime"] = pd.to_datetime(time_data["access_datetime"], errors="coerce")

time_data["month"] = time_data["access_datetime"].dt.month_name()
time_data["weekday"] = time_data["access_datetime"].dt.day_name()
time_data["hour"] = time_data["access_datetime"].dt.hour

display(time_data.head(10))

,access_id,access_datetime,resource_type,session_minutes,month,weekday,hour
0,A00001,2025-03-19 23:30:00,Database,33.6,March,Wednesday,23.0
1,A00002,2025-03-26 14:18:00,Thesis,26.9,March,Wednesday,14.0
2,A00003,2025-01-27 07:54:00,Journal,60.3,January,Monday,7.0
3,A00004,2025-12-16 12:39:00,Journal,42.0,December,Tuesday,12.0
4,A00005,2025-09-20 11:53:00,Journal,8.0,September,Saturday,11.0
5,A00006,2025-09-26 01:14:00,Database,12.1,September,Friday,1.0
6,A00007,2025-10-23 08:50:00,E-Book,37.2,October,Thursday,8.0
7,A00008,2025-08-09 14:19:00,E-Book,10.5,August,Saturday,14.0
8,A00009,2025-07-04 12:27:00,Database,27.8,July,Friday,12.0
9,A00010,2025-06-04 10:51:00,E-Book,NaN,June,Wednesday,10.0


In [26]:
## Monthly Access Summary
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

monthly_summary = time_data["month"].value_counts().reindex(month_order).reset_index()
monthly_summary.columns = ["month", "access_count"]

display(monthly_summary)

,month,access_count
0,January,99
1,February,117
2,March,171
3,April,188
4,May,77
5,June,52
6,July,94
7,August,148
8,September,163
9,October,188


In [21]:
## 3.4 Device and Access Mode-based Extraction

In [22]:
access_data = df[
    ["access_id", "device", "access_mode", "user_type", "session_minutes"]
].copy()

access_summary = (
    access_data.groupby(["access_mode", "device"], dropna=False)
    .size()
    .reset_index(name="access_count")
)

display(access_summary)

,access_mode,device,access_count
0,On-Campus,Campus PC,156
1,On-Campus,Laptop,399
2,On-Campus,Mobile,322
3,On-Campus,NaN,14
4,Remote,Campus PC,108
5,Remote,Laptop,273
6,Remote,Mobile,221
7,Remote,NaN,7


In [20]:
## Remote Access Cases

In [29]:
remote_cases = access_data[access_data["access_mode"].astype(str).str.strip().eq("Remote")]
display(remote_cases)

,access_id,device,access_mode,user_type,session_minutes
0,A00001,Laptop,Remote,Undergraduate,33.6
1,A00002,Laptop,Remote,Undergraduate,26.9
2,A00003,Campus PC,Remote,Faculty,60.3
4,A00005,Laptop,Remote,Postgraduate,8.0
5,A00006,Campus PC,Remote,Faculty,12.1
...,...,...,...,...,...
1484,A01485,Mobile,Remote,Undergraduate,12.8
1486,A01487,Mobile,Remote,Postgraduate,9.7
1490,A01491,Laptop,Remote,Undergraduate,26.5
1491,A01492,Mobile,Remote,Undergraduate,22.4


In [27]:
## 3.5 Engagement Extraction (Downloads, Ratings, Session Time)

In [28]:
engagement_data = df[
    ["access_id", "resource_id", "resource_type",
     "downloaded", "rating", "session_minutes"]
].copy()

display(engagement_data.head(10))

,access_id,resource_id,resource_type,downloaded,rating,session_minutes
0,A00001,R070,Database,0,4.0,33.6
1,A00002,R068,Thesis,1,4.0,26.9
2,A00003,R023,Journal,0,5.0,60.3
3,A00004,R049,Journal,1,2.0,42.0
4,A00005,R056,Journal,0,4.0,8.0
5,A00006,R070,Database,1,2.0,12.1
6,A00007,R055,E-Book,1,5.0,37.2
7,A00008,R038,E-Book,1,4.0,10.5
8,A00009,R036,Database,0,3.0,27.8
9,A00010,R038,E-Book,1,2.0,NaN


In [30]:
## Accesses with Downloads

In [31]:
download_cases = engagement_data[
    engagement_data["downloaded"].notna() & (engagement_data["downloaded"] == 1)
]
display(download_cases)

,access_id,resource_id,resource_type,downloaded,rating,session_minutes
1,A00002,R068,Thesis,1,4.0,26.9
3,A00004,R049,Journal,1,2.0,42.0
5,A00006,R070,Database,1,2.0,12.1
6,A00007,R055,E-Book,1,5.0,37.2
7,A00008,R038,E-Book,1,4.0,10.5
...,...,...,...,...,...,...
1485,A01486,R006,E-Book,1,4.0,20.9
1488,A01489,R003,Database,1,3.0,7.9
1493,A01494,R024,Journal,1,1.0,999.0
1497,A01498,R021,Database,1,5.0,49.8


In [36]:
## Accesses with Ratings

In [35]:
rated_cases = engagement_data[engagement_data["rating"].notna()]
display(rated_cases)

,access_id,resource_id,resource_type,downloaded,rating,session_minutes
0,A00001,R070,Database,0,4.0,33.6
1,A00002,R068,Thesis,1,4.0,26.9
2,A00003,R023,Journal,0,5.0,60.3
3,A00004,R049,Journal,1,2.0,42.0
4,A00005,R056,Journal,0,4.0,8.0
...,...,...,...,...,...,...
1495,A01496,R067,E-Book,0,4.0,10.8
1496,A01497,R070,Database,0,5.0,25.7
1497,A01498,R021,Database,1,5.0,49.8
1498,A01499,R049,Journal,0,5.0,44.4


In [34]:
## Accesses with Recorded Engagement

In [33]:
engaged_cases = engagement_data[
    (engagement_data["downloaded"].fillna(0) > 0) |
    (engagement_data["rating"].notna())
]
display(engaged_cases)

,access_id,resource_id,resource_type,downloaded,rating,session_minutes
0,A00001,R070,Database,0,4.0,33.6
1,A00002,R068,Thesis,1,4.0,26.9
2,A00003,R023,Journal,0,5.0,60.3
3,A00004,R049,Journal,1,2.0,42.0
4,A00005,R056,Journal,0,4.0,8.0
...,...,...,...,...,...,...
1495,A01496,R067,E-Book,0,4.0,10.8
1496,A01497,R070,Database,0,5.0,25.7
1497,A01498,R021,Database,1,5.0,49.8
1498,A01499,R049,Journal,0,5.0,44.4


In [32]:
# 4. Data Validation
### The validation is performed on the original dataset before cleaning.

In [43]:
## 4.1 Missing Values and Missing Percentage

In [42]:
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

display(
    missing_report[missing_report["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

,missing_count,missing_percentage
rating,89,5.93
session_minutes,67,4.47
device,21,1.40
publisher,16,1.07


In [40]:
## 4.2 Duplicate Records and Access IDs

In [41]:
print("Duplicate records:", df.duplicated().sum())
print("Duplicate access IDs:", df["access_id"].duplicated().sum())

Duplicate records: 29
Duplicate access IDs: 29


In [39]:
## 4.3 Validate ID Formats

In [38]:
valid_access = df["access_id"].astype(str).str.fullmatch(r"A\d{5}")
valid_user = df["user_id"].astype(str).str.fullmatch(r"U\d{4}")
valid_resource = df["resource_id"].astype(str).str.fullmatch(r"R\d{3}")

print("Invalid access IDs:", (~valid_access).sum())
print("Invalid user IDs:", (~valid_user).sum())
print("Invalid resource IDs:", (~valid_resource).sum())

Invalid access IDs: 0
Invalid user IDs: 0
Invalid resource IDs: 0


In [37]:
## 4.4 Validate Date Format and Range

In [48]:
parsed_dates = pd.to_datetime(df["access_datetime"], errors="coerce")

print("Invalid date values:", parsed_dates.isna().sum())

outside_range = (
    parsed_dates.notna() &
    ((parsed_dates.dt.year < 2020) | (parsed_dates.dt.year > 2026))
)
print("Dates outside expected range:", outside_range.sum())

display(df.loc[parsed_dates.isna(), ["access_id", "access_datetime"]].head(10))

Invalid date values: 15
Dates outside expected range: 0


,access_id,access_datetime
22,A00023,invalid
177,A00178,invalid
278,A00279,invalid
279,A00280,invalid
316,A00317,invalid
496,A00497,invalid
593,A00594,invalid
649,A00650,invalid
704,A00705,invalid
776,A00777,invalid


In [47]:
## 4.5 Validate Categorical Values

In [46]:
expected_values = {
    "user_type": {"Undergraduate", "Postgraduate", "Faculty", "Researcher"},
    "department": {"CSE", "ECE", "Mechanical", "Civil", "Management", "Sciences"},
    "resource_type": {"E-Book", "Journal", "Database", "Video", "Thesis"},
    "device": {"Mobile", "Laptop", "Campus PC"},
    "access_mode": {"On-Campus", "Remote"}
}

for column, allowed_values in expected_values.items():
    invalid = ~df[column].isin(allowed_values) & df[column].notna()
    print(f"{column} - invalid values:", invalid.sum())

    if invalid.any():
        display(df.loc[invalid, column].value_counts())

user_type - invalid values: 26


,count
user_type,
UNDERGRADUATE,15
POSTGRADUATE,6
FACULTY,4
RESEARCHER,1


department - invalid values: 50


,count
department,
cse,29
ece,21


resource_type - invalid values: 0
device - invalid values: 0
access_mode - invalid values: 0


In [45]:
## 4.6 Check Blank Text Values and Extra Whitespace

In [44]:
text_columns = df.select_dtypes(include="object").columns

blank_report = {
    column: df[column].fillna("").astype(str).str.strip().eq("").sum()
    for column in text_columns
}

whitespace_report = {
    column: (
        df[column].dropna().astype(str).str.len() !=
        df[column].dropna().astype(str).str.strip().str.len()
    ).sum()
    for column in text_columns
}

print("Blank text values:")
display(pd.Series(blank_report, name="blank_values")[lambda x: x > 0])

print("Values with extra whitespace:")
display(pd.Series(whitespace_report, name="extra_whitespace")[lambda x: x > 0])

Blank text values:


,blank_values
publisher,16
device,21


Values with extra whitespace:


,extra_whitespace
department,29


In [52]:
## 4.7 Validate Numerical Data

In [51]:
numeric_columns = ["publication_year", "session_minutes", "downloaded", "rating"]

print("Data types:")
display(df[numeric_columns].dtypes)

negative_values = {column: (df[column] < 0).sum() for column in numeric_columns}

print("Negative values:")
display(pd.Series(negative_values, name="negative_count"))

Data types:


,0
publication_year,int64
session_minutes,float64
downloaded,int64
rating,float64


Negative values:


,negative_count
publication_year,0
session_minutes,24
downloaded,0
rating,0


In [50]:
## 4.8 Check Numerical Consistency

In [49]:
invalid_session = df["session_minutes"].notna() & (
    (df["session_minutes"] <= 0) | (df["session_minutes"] > 480)
)
invalid_rating = df["rating"].notna() & ~df["rating"].between(1, 5)
invalid_download = df["downloaded"].notna() & ~df["downloaded"].isin([0, 1])
invalid_year = df["publication_year"].notna() & ~df["publication_year"].between(1900, 2026)

print("Sessions that are zero/negative or longer than 8 hours:", invalid_session.sum())
print("Ratings outside 1-5:", invalid_rating.sum())
print("Downloaded values other than 0/1:", invalid_download.sum())
print("Publication years outside 1900-2026:", invalid_year.sum())

if invalid_session.any():
    display(df.loc[invalid_session, ["access_id", "resource_type", "session_minutes"]].head(20))

Sessions that are zero/negative or longer than 8 hours: 39
Ratings outside 1-5: 0
Downloaded values other than 0/1: 0
Publication years outside 1900-2026: 0


,access_id,resource_type,session_minutes
23,A00024,Database,999.0
24,A00025,E-Book,999.0
34,A00035,E-Book,-5.0
110,A00111,E-Book,-5.0
146,A00147,E-Book,-5.0
151,A00152,E-Book,999.0
175,A00176,Journal,999.0
293,A00294,Journal,999.0
321,A00322,Journal,-5.0
346,A00347,Journal,-5.0


In [60]:
## 4.9 Validate Resource Information Consistency

In [59]:
inconsistent = (
    df.groupby("resource_id")
      .agg(titles=("resource_title", "nunique"),
           types=("resource_type", "nunique"),
           publishers=("publisher", "nunique"))
)

print("Resources with more than one title/type/publisher:")
display(inconsistent[(inconsistent > 1).any(axis=1)])

Resources with more than one title/type/publisher:


,titles,types,publishers
resource_id,,,


In [58]:
## 4.10 Validate Resource Titles and Publishers

In [57]:
blank_title = df["resource_title"].isna() | df["resource_title"].astype(str).str.strip().eq("")
blank_publisher = df["publisher"].isna() | df["publisher"].astype(str).str.strip().eq("")

print("Blank resource titles:", blank_title.sum())
print("Blank publishers:", blank_publisher.sum())

Blank resource titles: 0
Blank publishers: 16


In [56]:
# 5. Data Cleaning
## The original df is kept unchanged. Cleaning is performed on a separate copy.

In [55]:
cleaned_df = df.copy()

In [54]:
## 5.1 Remove Duplicate Records

In [61]:
before = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates().copy()

print("Duplicate records removed:", before - len(cleaned_df))

Duplicate records removed: 29


In [64]:
## 5.2 Standardize Text and Categorical Values

In [65]:
text_columns = cleaned_df.select_dtypes(include="object").columns

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

for column in ["user_type", "device", "access_mode"]:
    cleaned_df[column] = cleaned_df[column].str.title()

cleaned_df["department"] = cleaned_df["department"].apply(
    lambda x: x if pd.isna(x) else (x.upper() if x.lower() in ("cse", "ece") else x.title())
)

print("Text and categorical formatting standardized.")
display(cleaned_df["department"].value_counts())
display(cleaned_df["user_type"].value_counts())

Text and categorical formatting standardized.


,count
department,
CSE,289
ECE,260
Mechanical,247
Management,240
Sciences,219
Civil,216


,count
user_type,
Undergraduate,791
Postgraduate,371
Faculty,175
Researcher,134


In [63]:
## 5.3 Standardize Resource Type, Publisher and Device

In [62]:
cleaned_df["resource_type"] = cleaned_df["resource_type"].replace({"E-book": "E-Book", "Ebook": "E-Book"})

cleaned_df["publisher"] = cleaned_df["publisher"].fillna("Unknown")
cleaned_df["device"] = cleaned_df["device"].fillna("Unknown")

print("Resource type, publisher and device standardized.")

Resource type, publisher and device standardized.


In [71]:
## 5.4 Convert Date and Numerical Columns

In [70]:
cleaned_df["access_datetime"] = pd.to_datetime(cleaned_df["access_datetime"], errors="coerce")

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(cleaned_df[column], errors="coerce")

print("Date and numerical columns converted.")

Date and numerical columns converted.


In [69]:
## 5.5 Handle Invalid Dates
### Rows whose access date cannot be read cannot be placed in any month, weekday or hour, so they are removed.

In [68]:
before = len(cleaned_df)

cleaned_df = cleaned_df.dropna(subset=["access_datetime"]).copy()

print("Rows removed because of invalid dates:", before - len(cleaned_df))

Rows removed because of invalid dates: 0


In [67]:
## 5.6 Handle Invalid and Missing Session Time
### Zero, negative or over-8-hour sessions are set to missing, then filled with the median for the same resource type.

In [66]:
invalid_session = (cleaned_df["session_minutes"] <= 0) | (cleaned_df["session_minutes"] > 480)
print("Invalid session values set to missing:", invalid_session.sum())

cleaned_df.loc[invalid_session, "session_minutes"] = pd.NA
cleaned_df["session_minutes"] = pd.to_numeric(cleaned_df["session_minutes"])

cleaned_df["session_minutes"] = cleaned_df.groupby("resource_type")["session_minutes"] \
    .transform(lambda s: s.fillna(s.median()))

print("Missing session values after filling:", cleaned_df["session_minutes"].isna().sum())

Invalid session values set to missing: 39
Missing session values after filling: 0


In [78]:
## 5.7 Handle Missing Ratings
### Missing ratings are retained because the user may not have rated the resource.

In [77]:
print("Missing ratings retained:", cleaned_df["rating"].isna().sum())

Missing ratings retained: 89


In [76]:
## 5.8 Create Date Features

In [75]:
cleaned_df["month"] = cleaned_df["access_datetime"].dt.month_name()
cleaned_df["month_no"] = cleaned_df["access_datetime"].dt.month
cleaned_df["weekday"] = cleaned_df["access_datetime"].dt.day_name()
cleaned_df["hour"] = cleaned_df["access_datetime"].dt.hour

display(cleaned_df[["access_datetime", "month", "weekday", "hour"]].head())

,access_datetime,month,weekday,hour
0,2025-03-19 23:30:00,March,Wednesday,23.0
1,2025-03-26 14:18:00,March,Wednesday,14.0
2,2025-01-27 07:54:00,January,Monday,7.0
3,2025-12-16 12:39:00,December,Tuesday,12.0
4,2025-09-20 11:53:00,September,Saturday,11.0


In [74]:
#6. Final Validation

In [73]:
## 6.1 Missing Values After Cleaning

In [72]:
final_missing = pd.DataFrame({
    "missing_count": cleaned_df.isnull().sum(),
    "missing_percentage": (cleaned_df.isnull().mean() * 100).round(2)
})

display(
    final_missing[final_missing["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

,missing_count,missing_percentage
rating,89,6.05
access_datetime,15,1.02


In [88]:
## 6.2 Duplicate Records After Cleaning

In [87]:
print("Duplicate records:", cleaned_df.duplicated().sum())
print("Duplicate access IDs:", cleaned_df["access_id"].duplicated().sum())

Duplicate records: 0
Duplicate access IDs: 0


In [86]:
## 6.3 Data Types After Cleaning

In [85]:
display(cleaned_df.dtypes)

,0
access_id,object
user_id,object
user_type,object
department,object
resource_id,object
resource_title,object
resource_type,object
publisher,object
publication_year,int64
access_datetime,datetime64[ns]


In [84]:
## 6.4 Final Dataset Dimensions

In [83]:
print("Final rows:", cleaned_df.shape[0])
print("Final columns:", cleaned_df.shape[1])

Final rows: 1471
Final columns: 19


In [82]:
## 6.5 Display the Cleaned Dataset

In [81]:
display(cleaned_df.head(10))

,access_id,user_id,user_type,department,resource_id,resource_title,resource_type,publisher,publication_year,access_datetime,session_minutes,device,downloaded,rating,access_mode,month,month_no,weekday,hour
0,A00001,U1675,Undergraduate,Mechanical,R070,SpringerLink Vol.1,Database,IEEE,2015,2025-03-19 23:30:00,33.60,Laptop,0,4.0,Remote,March,3.0,Wednesday,23.0
1,A00002,U1448,Undergraduate,ECE,R068,PhD Thesis Vol.3,Thesis,Wiley,2008,2025-03-26 14:18:00,26.90,Laptop,1,4.0,Remote,March,3.0,Wednesday,14.0
2,A00003,U1015,Faculty,Civil,R023,Finance Vol.2,Journal,Pearson,2005,2025-01-27 07:54:00,60.30,Campus Pc,0,5.0,Remote,January,1.0,Monday,7.0
3,A00004,U1124,Undergraduate,Sciences,R049,Finance Vol.2,Journal,NPTEL,2011,2025-12-16 12:39:00,42.00,Laptop,1,2.0,On-Campus,December,12.0,Tuesday,12.0
4,A00005,U1298,Postgraduate,Sciences,R056,Signal Processing Vol.5,Journal,Wiley,2021,2025-09-20 11:53:00,8.00,Laptop,0,4.0,Remote,September,9.0,Saturday,11.0
5,A00006,U1026,Faculty,Mechanical,R070,SpringerLink Vol.1,Database,IEEE,2015,2025-09-26 01:14:00,12.10,Campus Pc,1,2.0,Remote,September,9.0,Friday,1.0
6,A00007,U1117,Postgraduate,CSE,R055,Data Science Vol.1,E-Book,Springer,2016,2025-10-23 08:50:00,37.20,Laptop,1,5.0,On-Campus,October,10.0,Thursday,8.0
7,A00008,U1554,Undergraduate,Sciences,R038,Marketing Vol.2,E-Book,Taylor & Francis,2017,2025-08-09 14:19:00,10.50,Unknown,1,4.0,Remote,August,8.0,Saturday,14.0
8,A00009,U1099,Faculty,CSE,R036,JSTOR Vol.3,Database,McGraw Hill,2010,2025-07-04 12:27:00,27.80,Mobile,0,3.0,Remote,July,7.0,Friday,12.0
9,A00010,U1116,Postgraduate,Civil,R038,Marketing Vol.2,E-Book,Taylor & Francis,2017,2025-06-04 10:51:00,25.85,Laptop,1,2.0,Remote,June,6.0,Wednesday,10.0


In [80]:
## 6.6 Save the Cleaned Dataset

In [79]:
cleaned_df.to_csv("library_usage_cleaned.csv", index=False)

print("Cleaned dataset saved as library_usage_cleaned.csv")

from google.colab import files
files.download("library_usage_cleaned.csv")

Cleaned dataset saved as library_usage_cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [89]:
# End of Data Extraction, Validation and Cleaning

In [91]:
### The cleaned dataset can now be passed to the next stages of the project for Data Aggregation, Representation, Analysis, Visualization and Results Interpretation.